In [ ]:
!pip install bitsandbytes llama-cpp-python datasets

In [1]:
import time
from copy import deepcopy
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer

import warnings
warnings.filterwarnings("ignore")

# 1) Foundations: Numeric Data Types & Precision

<img src="https://substackcdn.com/image/fetch/$s_!-PZj!,w_1456,c_limit,f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fsubstack-post-media.s3.amazonaws.com%2Fpublic%2Fimages%2F8ffa0c54-88bf-45c1-8636-bdb097bb8e6b_1172x848.png" width=500>

Before we can compress a model, we need to understand HOW numbers are stored.
Neural network weights are floating-point numbers. Quantization maps them to a lower-precision representation (fewer bits) to save memory and speed up inference, at the cost of some precision loss.

### Key Data Types:
- **FP32** — 32-bit float (1 sign + 8 exponent + 23 mantissa) — full precision
- **FP16** — 16-bit float (1 sign + 5 exponent + 10 mantissa) — half precision
- **BF16** — 16-bit bfloat (1 sign + 8 exponent + 7 mantissa) — same range as FP32, less precision
- **INT8** — 8-bit integer (−128 to 127 or 0 to 255)
- **INT4** — 4-bit integer (−8 to 7 or 0 to 15)

### Memory Rule of Thumb:
- A model with $N$ parameters in FP32 uses $N \times 4$ bytes.
- Casting to BF16/FP16 halves that to $N \times 2$ bytes.
- Quantizing to INT8 quarters it to $N \times 1$ byte.
- Quantizing to INT4 reduces it to $N \times 0.5$ bytes.

In [2]:
# Understand the numeric limits of each dtype the hardware supports.
print("\n--- Data-type ranges ---")
print("uint8 : ", torch.iinfo(torch.uint8))  # 0 .. 255
print("int8  : ", torch.iinfo(torch.int8))  # -128 .. 127
print("bfloat16: ", torch.finfo(torch.bfloat16))
print("float16 : ", torch.finfo(torch.float16))
print("float32 : ", torch.finfo(torch.float32))

# Store the same irrational value (1/3) in different precisions and observe
# how many decimal digits are faithfully preserved.
value = 1 / 3
tensor_fp64 = torch.tensor(value, dtype=torch.float64)
tensor_fp32 = torch.tensor(value, dtype=torch.float32)
tensor_fp16 = torch.tensor(value, dtype=torch.float16)
tensor_bf16 = torch.tensor(value, dtype=torch.bfloat16)

# Precision of 1/3 across dtypes
print(f"fp64 : {tensor_fp64.item():.60f}")
print(f"fp32 : {tensor_fp32.item():.60f}")
print(f"fp16 : {tensor_fp16.item():.60f}")
print(f"bf16 : {tensor_bf16.item():.60f}")

# BF16 keeps the same exponent range as FP32
# it can represent very large and very small values, but introduces rounding error

# Generate random FP32 values
# (num_samples,) = (1000,)
tensor_fp32 = torch.rand(1000, dtype=torch.float32)

# Cast down to BF16 — some precision is lost
# (num_samples,) = (1000,)
tensor_bf16 = tensor_fp32.to(dtype=torch.bfloat16)

print("FP32 sample: ", tensor_fp32[:5])
print("BF16 sample: ", tensor_bf16[:5])

mean_error = torch.abs(tensor_fp32 - tensor_bf16.float()).mean().item()
max_error = torch.abs(tensor_fp32 - tensor_bf16.float()).max().item()
print(f"Mean absolute error: {mean_error:.8f}")
print(f"Max  absolute error: {max_error:.8f}")


--- Data-type ranges ---
uint8 :  iinfo(min=0, max=255, dtype=uint8)
int8  :  iinfo(min=-128, max=127, dtype=int8)
bfloat16:  finfo(resolution=0.01, min=-3.38953e+38, max=3.38953e+38, eps=0.0078125, smallest_normal=1.17549e-38, tiny=1.17549e-38, dtype=bfloat16)
float16 :  finfo(resolution=0.001, min=-65504, max=65504, eps=0.000976562, smallest_normal=6.10352e-05, tiny=6.10352e-05, dtype=float16)
float32 :  finfo(resolution=1e-06, min=-3.40282e+38, max=3.40282e+38, eps=1.19209e-07, smallest_normal=1.17549e-38, tiny=1.17549e-38, dtype=float32)
fp64 : 0.333333333333333314829616256247390992939472198486328125000000
fp32 : 0.333333343267440795898437500000000000000000000000000000000000
fp16 : 0.333251953125000000000000000000000000000000000000000000000000
bf16 : 0.333984375000000000000000000000000000000000000000000000000000
FP32 sample:  tensor([0.3182, 0.4685, 0.5806, 0.7282, 0.6829])
BF16 sample:  tensor([0.3184, 0.4688, 0.5820, 0.7266, 0.6836], dtype=torch.bfloat16)
Mean absolute error: 0.

In [4]:
def print_param_dtype(model):
    """Print name and dtype of every parameter in a model."""
    for name, param in model.named_parameters():
        print(f"  {name:40s} → {param.dtype}")


class DummyModel(nn.Module):
    """
    A minimal transformer-like block: embedding → 2×(linear + layernorm) → head.
    Used purely to demonstrate dtype casting and quantization effects.
    """

    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(2, 2)
        self.linear_1 = nn.Linear(2, 2)
        self.layernorm_1 = nn.LayerNorm(2)
        self.linear_2 = nn.Linear(2, 2)
        self.layernorm_2 = nn.LayerNorm(2)
        self.head = nn.Linear(2, 2)

    def forward(self, x):
        # Embed token ids
        # (batch_num, seq_len) → (batch_num, seq_len, embed_dim=2)
        x = self.token_embedding(x)

        # First linear projection
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, hidden_dim=2)
        x = self.linear_1(x)
        x = self.layernorm_1(x)
        x = torch.relu(x)

        # Second linear projection
        # (batch_num, seq_len, hidden_dim) → (batch_num, seq_len, hidden_dim)
        x = self.linear_2(x)
        x = self.layernorm_2(x)
        x = torch.relu(x)

        # Classification / language-modelling head
        # (batch_num, seq_len, hidden_dim) → (batch_num, seq_len, vocab_size=2)
        x = self.head(x)
        return x


# Default model is FP32
model_fp32 = DummyModel()
print_param_dtype(model_fp32)

# .half() casts every parameter to FP16
model_fp16 = DummyModel().half()
print_param_dtype(model_fp16)

dummy_input = torch.LongTensor([[1, 0], [0, 1]])

# FP32 forward pass
# (batch_num=2, seq_len=2) → (batch_num=2, seq_len=2, vocab_size=2)
logits_fp32 = model_fp32(dummy_input)

# FP16 and BF16 copies of the SAME weights
model_fp16 = deepcopy(model_fp32).half()
# (batch_num=2, seq_len=2) → (batch_num=2, seq_len=2, vocab_size=2)
logits_fp16 = model_fp16(dummy_input)

model_bf16 = deepcopy(model_fp32).to(torch.bfloat16)
# (batch_num=2, seq_len=2) → (batch_num=2, seq_len=2, vocab_size=2)
logits_bf16 = model_bf16(dummy_input)

# FP16 keeps more mantissa bits than BF16, so it stays closer to FP32 (as long as values fit its smaller range)
for label, logits in [("FP16", logits_fp16), ("BF16", logits_bf16)]:
    diff = torch.abs(logits.float() - logits_fp32)
    print(f"\nFP32 vs {label} logit difference → Mean: {diff.mean():.6f} | Max: {diff.max():.6f}")

  token_embedding.weight                   → torch.float32
  linear_1.weight                          → torch.float32
  linear_1.bias                            → torch.float32
  layernorm_1.weight                       → torch.float32
  layernorm_1.bias                         → torch.float32
  linear_2.weight                          → torch.float32
  linear_2.bias                            → torch.float32
  layernorm_2.weight                       → torch.float32
  layernorm_2.bias                         → torch.float32
  head.weight                              → torch.float32
  head.bias                                → torch.float32
  token_embedding.weight                   → torch.float16
  linear_1.weight                          → torch.float16
  linear_1.bias                            → torch.float16
  layernorm_1.weight                       → torch.float16
  layernorm_1.bias                         → torch.float16
  linear_2.weight                          → torch.float

In [5]:
# Memory Footprint Comparison

# Compare the memory used by BLIP (image-captioning model) in FP32 vs BF16.
from transformers import BlipForConditionalGeneration

model_name = "Salesforce/blip-image-captioning-base"

# Load in FP32 (default)
model_fp32_blip = BlipForConditionalGeneration.from_pretrained(model_name)
fp32_bytes = model_fp32_blip.get_memory_footprint()

# Load in BF16
model_bf16_blip = BlipForConditionalGeneration.from_pretrained(
    model_name, dtype=torch.bfloat16
)
bf16_bytes = model_bf16_blip.get_memory_footprint()

print(f"FP32 : {fp32_bytes / 1e6:.1f} MB")
print(f"BF16 : {bf16_bytes / 1e6:.1f} MB")
print(f"Ratio: {bf16_bytes / fp32_bytes * 100:.1f}%  (≈ 50% is expected)")

# Clean up GPU/CPU memory
del model_fp32_blip, model_bf16_blip

FP32 : 895.9 MB
BF16 : 447.9 MB
Ratio: 50.0%  (≈ 50% is expected)


# 2) Post-Training Quantization (PTQ)

## 2.1 Asymmetric (Zero-Point) Quantization

The simplest form of quantization: map a floating-point range `[min, max]` linearly to the integer range `[0, 255]` (uint8).

### Formulas:
- **scale** $= \frac{\text{max\_val} - \text{min\_val}}{2^{\text{bits}} - 1}$
- **zero_point** $= \text{min\_val}$
- **quantize**: $q = \text{clamp}(\text{round}(\frac{x - \text{zero\_point}}{\text{scale}}), 0, 2^{\text{bits}}-1)$
- **dequantize**: $\hat{x} = q \times \text{scale} + \text{zero\_point}$

It is called "Asymmetric" because the zero of the float range does NOT necessarily map to the zero of the int range.

In [6]:
# Load GPT-2 as the model to quantize
model_name = "openai-community/gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Set pad token (GPT-2 doesn't define one by default)
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = model.config.eos_token_id

print(f"GPT-2 FP32 memory: {model.get_memory_footprint() / 1e6:.1f} MB")

GPT-2 FP32 memory: 497.8 MB


In [7]:
# === Quantize a Single Tensor ===
def quantize_asymmetric(t, bits=8):
    """
    Asymmetric (zero-point) quantization of a float tensor to unsigned int.

    Args:
        t:    torch.Tensor — the floating-point tensor to quantize
        bits: int — target bit-width (default 8 → uint8, range 0..255)

    Returns:
        t_quant: torch.Tensor (uint8) — quantized tensor
        state:   tuple(scale, zero_point) — needed for dequantization
    """
    qmin, qmax = 0, 2**bits - 1

    min_val, max_val = t.min(), t.max()

    # Scale maps the float range to the int range
    scale = (max_val - min_val) / (qmax - qmin)

    # Zero-point: the float value that maps to integer 0
    zero_point = min_val

    # Quantize: shift, scale, round, clamp
    t_quant = (t - zero_point) / scale
    t_quant = torch.clamp(torch.round(t_quant), qmin, qmax)
    t_quant = t_quant.to(torch.uint8)

    state = (scale, zero_point)
    return t_quant, state


def dequantize_asymmetric(t_quant, state):
    """
    Reverse the asymmetric quantization.

    Args:
        t_quant: torch.Tensor (uint8)
        state:   tuple(scale, zero_point)

    Returns:
        torch.Tensor (float32) — approximate reconstruction of the original
    """
    scale, zero_point = state
    return t_quant.to(torch.float32) * scale + zero_point


# Grab one weight tensor from GPT-2's first attention block
# (model_dim, 3 * model_dim) = (768, 2304) for GPT-2
t_original = model.transformer.h[0].attn.c_attn.weight.data
print(f"\nOriginal weight shape : {t_original.shape}")
print(f"Original dtype       : {t_original.dtype}")
print(f"Value range          : [{t_original.min():.6f}, {t_original.max():.6f}]")

# Quantize
t_quant, state = quantize_asymmetric(t_original)
print(f"\nQuantized dtype       : {t_quant.dtype}")
print(f"Quantized range      : [{t_quant.min()}, {t_quant.max()}]")
print(f"Scale                : {state[0]:.8f}")
print(f"Zero-point           : {state[1]:.8f}")

# Dequantize and measure reconstruction error
t_recon = dequantize_asymmetric(t_quant, state)
abs_error = torch.abs(t_original - t_recon)
print(
    f"\nReconstruction error → Mean: {abs_error.mean():.8f} | Max: {abs_error.max():.8f}"
)


Original weight shape : torch.Size([768, 2304])
Original dtype       : torch.float32
Value range          : [-2.843634, 2.795630]

Quantized dtype       : torch.uint8
Quantized range      : [0, 255]
Scale                : 0.02211476
Zero-point           : -2.84363437

Reconstruction error → Mean: 0.00553088 | Max: 0.01105749


In [8]:
# === Quantize the Entire Model ===
def quantize_model_asymmetric(model, bits=8):
    """Quantize every parameter in the model using asymmetric quantization."""
    states = {}
    for name, param in model.named_parameters():
        param.requires_grad = False
        param.data, state = quantize_asymmetric(param.data, bits)
        states[name] = state
    return model, states


def dequantize_model(model, states):
    """Dequantize every parameter back to float32."""
    for name, param in model.named_parameters():
        param.data = dequantize_asymmetric(param.data, states[name])
    return model


# Fix dtype property so HF reports correct memory even after param dtype changes
from transformers.models.gpt2.modeling_gpt2 import GPT2Model

GPT2Model.dtype = property(lambda self: torch.float32)

quant_model, states = quantize_model_asymmetric(model)
print(f"\nQuantized GPT-2 memory : {quant_model.get_memory_footprint() / 1e6:.1f} MB")

dequant_model = dequantize_model(quant_model, states)
print(f"Dequantized GPT-2 memory: {dequant_model.get_memory_footprint() / 1e6:.1f} MB")

del model, quant_model, dequant_model  # free memory


Quantized GPT-2 memory : 124.4 MB
Dequantized GPT-2 memory: 497.8 MB


## 2.2 Symmetric (Absmax) Quantization

Symmetric quantization maps the float range `[−|max|, +|max|]` symmetrically to `[−127, +127]` (int8).

### Formula:
- **scale** $= \frac{\max(|t|)}{127}$
- **quantize**: $q = \text{clamp}(\text{round}(\frac{t}{\text{scale}}), −127, 127)$
- **dequantize**: $\hat{x} = q \times \text{scale}$


**Pros:** Simpler (no zero-point), plays well with symmetric weight distributions.

**Cons:** Wastes range if the distribution is heavily skewed.

In [9]:
def quantize_symmetric(t, bits=8):
    """
    Symmetric (absmax) quantization to signed int8.

    Args:
        t:    torch.Tensor (float)
        bits: int — target bit-width

    Returns:
        t_quant: torch.Tensor (int8)
        scale:   float — single scale factor for dequantization
    """
    qmax = 2 ** (bits - 1) - 1  # 127 for 8-bit

    # Scale is determined by the absolute maximum value
    scale = t.abs().max() / qmax

    t_quant = torch.clamp(torch.round(t / scale), -qmax, qmax)
    t_quant = t_quant.to(torch.int8)
    return t_quant, scale


def dequantize_symmetric(t_quant, scale):
    """Reverse symmetric quantization."""
    return t_quant.to(torch.float32) * scale


# Demonstrate on a small random tensor
t_demo = torch.randn(4, 4)
print(f"Original:\n{t_demo}")

t_q, s = quantize_symmetric(t_demo)
print(f"\nQuantized (int8):\n{t_q}")
print(f"Scale: {s:.6f}")

t_r = dequantize_symmetric(t_q, s)
print(f"\nReconstructed:\n{t_r}")
print(f"Max error: {(t_demo - t_r).abs().max():.6f}")

Original:
tensor([[-0.6090, -2.6965,  1.2314,  0.2999],
        [-0.9618, -0.9075,  1.3685,  0.0126],
        [-1.0992, -0.2383,  0.5685, -0.0261],
        [ 0.3460, -0.6695, -0.5843,  0.5839]])

Quantized (int8):
tensor([[ -29, -127,   58,   14],
        [ -45,  -43,   64,    1],
        [ -52,  -11,   27,   -1],
        [  16,  -32,  -28,   28]], dtype=torch.int8)
Scale: 0.021232

Reconstructed:
tensor([[-0.6157, -2.6965,  1.2315,  0.2972],
        [-0.9554, -0.9130,  1.3588,  0.0212],
        [-1.1041, -0.2336,  0.5733, -0.0212],
        [ 0.3397, -0.6794, -0.5945,  0.5945]])
Max error: 0.010613


## 2.3 GPTQ: Generative Pre-Trained Transformer Quantization

### Key Ideas:
1. Based on Optimal Brain Quantization (OBQ), which uses second-order (Hessian) information to quantize weights one-at-a-time while compensating the remaining weights.
2. Processes weights column-by-column within each layer, updating the unquantized weights to compensate for the error introduced.
3. Uses a small calibration dataset (128–256 samples) to compute the Hessian ($H = 2 X^T X$ from layer inputs).
4. Achieves 4-bit weight quantization with minimal perplexity degradation.

### Strengths vs Weaknesses:
- **Strengths:** First method to achieve 4-bit LLM quantization at scale. Excellent accuracy retention. Well-supported (HF, vLLM, TensorRT-LLM).
- **Weaknesses:** Slow quantization (hours for 70B models). Accuracy depends on calibration data. Weight-only (activations remain in FP16). GPU-centric.

In [11]:
from datasets import load_dataset

# Reload GPT-2 (FP32) and build a calibration set + a held-out evaluation set from WikiText-2
gpt2_tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token
gpt2 = AutoModelForCausalLM.from_pretrained("openai-community/gpt2").eval()

wikitext = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
texts = [t for t in wikitext["text"] if len(t) > 500][:128]
# (num_texts=128, seq_len=128)
encoded = gpt2_tokenizer(
    texts, return_tensors="pt", truncation=True, max_length=128, padding="max_length"
)
calib_batch = {k: v[:64] for k, v in encoded.items()}  # used to collect activations
eval_batch = {k: v[64:] for k, v in encoded.items()}  # used only for perplexity

# Every linear layer inside the 12 transformer blocks (GPT-2 uses Conv1D: y = x @ W + b, W is (in, out))
linear_names = [
    f"transformer.h.{i}.{name}"
    for i in range(12)
    for name in ["attn.c_attn", "attn.c_proj", "mlp.c_fc", "mlp.c_proj"]
]


@torch.no_grad()
def capture_layer_inputs(model, batch):
    """Record the input activations X of every linear layer during one forward pass."""
    inputs, hooks = {}, []
    for name in linear_names:
        layer = model.get_submodule(name)
        hooks.append(
            layer.register_forward_hook(
                lambda m, inp, out, name=name: inputs.__setitem__(name, inp[0])
            )
        )
    model(**batch)
    for h in hooks:
        h.remove()

    # Drop padding positions
    # (batch_num, seq_len, in_dim) -> (num_tokens, in_dim)
    mask = batch["attention_mask"].reshape(-1).bool()
    return {n: x.reshape(-1, x.shape[-1])[mask] for n, x in inputs.items()}


@torch.no_grad()
def perplexity(model, batch):
    labels = batch["input_ids"].masked_fill(batch["attention_mask"] == 0, -100)
    return torch.exp(model(**batch, labels=labels).loss).item()


def quantize_rtn(W, bits=4, group_size=128):
    """Round-to-nearest symmetric quantization with one scale per group of input weights.
    Returns the dequantized weight so it can be dropped straight back into the model."""
    qmax = 2 ** (bits - 1) - 1
    in_dim, out_dim = W.shape
    # (in_dim, out_dim) -> (out_dim, num_groups, group_size)
    W_groups = W.T.reshape(out_dim, -1, group_size)
    scale = W_groups.abs().amax(dim=-1, keepdim=True) / qmax
    W_q = torch.clamp(torch.round(W_groups / scale), -qmax - 1, qmax) * scale
    # (out_dim, num_groups, group_size) -> (in_dim, out_dim)
    return W_q.reshape(out_dim, in_dim).T


def quantize_model(model, quantize_fn):
    """Return a copy of the model with every linear weight replaced by quantize_fn(name, W)."""
    model_q = deepcopy(model)
    for name in linear_names:
        layer = model_q.get_submodule(name)
        layer.weight.data = quantize_fn(name, layer.weight.data)
    return model_q


layer_inputs = capture_layer_inputs(gpt2, calib_batch)
print(f"Calibration tokens per layer: {layer_inputs[linear_names[0]].shape[0]:,}")

ppl_results = {"FP32": perplexity(gpt2, eval_batch)}
for bits in [4, 3]:
    model_rtn = quantize_model(gpt2, lambda name, W: quantize_rtn(W, bits))
    ppl_results[f"RTN {bits}-bit"] = perplexity(model_rtn, eval_batch)

for k, v in ppl_results.items():
    print(f"{k:12s} perplexity: {v:.2f}")

Calibration tokens per layer: 8,107


FP32         perplexity: 56.51
RTN 4-bit    perplexity: 66.15
RTN 3-bit    perplexity: 223.56


In [12]:
def quantize_gptq(W, X, bits=4, group_size=128, damp=0.01):
    """
    GPTQ for one linear layer.

    Quantize the weights one input-column at a time. After each column is rounded,
    spread its rounding error onto the columns not yet quantized, weighted by the
    inverse Hessian, so the layer OUTPUT (X @ W) changes as little as possible.

    Args:
        W: (in_dim, out_dim) — FP32 weight
        X: (num_tokens, in_dim) — calibration inputs to this layer
    """
    qmax = 2 ** (bits - 1) - 1
    # Work in (out_dim, in_dim) so each column i is one input feature
    W = W.T.clone()
    in_dim = W.shape[1]

    # Hessian of the layer-wise squared error: H = 2 XᵀX
    # (in_dim, num_tokens) @ (num_tokens, in_dim) -> (in_dim, in_dim)
    H = 2 * X.T @ X
    # Dampening keeps H invertible
    H += damp * H.diag().mean() * torch.eye(in_dim)

    # Upper Cholesky factor of H⁻¹ — row i holds how column i's error propagates to later columns
    H_inv = torch.linalg.cholesky(
        torch.cholesky_inverse(torch.linalg.cholesky(H)), upper=True
    )

    for i in range(in_dim):
        # A new group starts: compute its scale from the (already error-compensated) weights
        if i % group_size == 0:
            # (out_dim,)
            scale = W[:, i : i + group_size].abs().amax(dim=1) / qmax

        # (out_dim,)
        w = W[:, i]
        q = torch.clamp(torch.round(w / scale), -qmax - 1, qmax) * scale

        # Rounding error, normalised by this column's curvature
        # (out_dim,)
        err = (w - q) / H_inv[i, i]
        W[:, i] = q

        # Compensate: push the error onto the remaining unquantized columns
        # (out_dim, 1) @ (1, in_dim - i - 1) -> (out_dim, in_dim - i - 1)
        W[:, i + 1 :] -= err.unsqueeze(1) @ H_inv[i, i + 1 :].unsqueeze(0)

    # (out_dim, in_dim) -> (in_dim, out_dim)
    return W.T


# Note: real GPTQ quantizes block-by-block and feeds each block the outputs of the
# already-quantized previous blocks. Here every layer uses the FP32 activations for simplicity.
start = time.time()
for bits in [4, 3]:
    model_gptq = quantize_model(
        gpt2, lambda name, W: quantize_gptq(W, layer_inputs[name], bits)
    )
    ppl_results[f"GPTQ {bits}-bit"] = perplexity(model_gptq, eval_batch)
print(f"GPTQ quantized 48 layers twice in {time.time() - start:.1f}s\n")

for bits in [4, 3]:
    print(
        f"{bits}-bit → RTN: {ppl_results[f'RTN {bits}-bit']:.2f} | "
        f"GPTQ: {ppl_results[f'GPTQ {bits}-bit']:.2f} | FP32: {ppl_results['FP32']:.2f}"
    )

GPTQ quantized 48 layers twice in 50.7s

4-bit → RTN: 66.15 | GPTQ: 60.21 | FP32: 56.51
3-bit → RTN: 223.56 | GPTQ: 85.77 | FP32: 56.51


## 2.4 AWQ: Activation-Aware Weight Quantization

### Key Insight:
Not all weights are equally important. A small fraction of weights correspond to large activation magnitudes ("salient channels"). Keeping those salient weights at higher precision dramatically reduces quantization error.

### How it works:
1. Run a small calibration set through the model and observe activation magnitudes per channel.
2. Identify salient weight channels (those producing large activations).
3. Apply per-channel scaling: multiply salient weights by a scale factor BEFORE quantization, then divide after dequantization.
4. Optimize the scale factors to minimize total quantization error.

In [13]:
def quantize_awq(W, X, bits=4, group_size=128, n_grid=20):
    """
    AWQ for one linear layer.

    Scale each input channel j by s_j = mean(|X_j|)^α before quantizing, then divide it back out:
        X @ W = (X / s) @ (s · W)
    Salient channels (large activations) get scaled UP, so they occupy more quantization levels
    and lose less precision. α is grid-searched to minimise the layer's output error.

    Args:
        W: (in_dim, out_dim) — FP32 weight
        X: (num_tokens, in_dim) — calibration inputs to this layer
    """
    # Average activation magnitude per input channel
    # (num_tokens, in_dim) -> (in_dim,)
    act_magnitude = X.abs().mean(dim=0)
    # (num_tokens, out_dim)
    reference_out = X @ W

    best_err, best_W = float("inf"), None
    for alpha in torch.linspace(0, 1, n_grid + 1):
        # (in_dim,)
        s = act_magnitude.pow(alpha)
        s = s / (s.max() * s.min()).sqrt()  # keep the scales centred around 1

        # Scale up salient rows, quantize, then undo the scale
        # (in_dim, out_dim)
        W_q = quantize_rtn(W * s.unsqueeze(1), bits, group_size) / s.unsqueeze(1)

        err = ((X @ W_q - reference_out) ** 2).mean().item()
        if err < best_err:
            best_err, best_W = err, W_q
    return best_W


start = time.time()
for bits in [4, 3]:
    model_awq = quantize_model(
        gpt2, lambda name, W: quantize_awq(W, layer_inputs[name], bits)
    )
    ppl_results[f"AWQ {bits}-bit"] = perplexity(model_awq, eval_batch)
print(f"AWQ searched scales for 48 layers twice in {time.time() - start:.1f}s\n")

print(f"{'':6s} {'RTN':>8s} {'AWQ':>8s} {'GPTQ':>8s}   (FP32 = {ppl_results['FP32']:.2f})")
for bits in [4, 3]:
    row = [ppl_results[f"{m} {bits}-bit"] for m in ["RTN", "AWQ", "GPTQ"]]
    print(f"{bits}-bit  " + " ".join(f"{v:8.2f}" for v in row))

del model_rtn, model_gptq, model_awq

AWQ searched scales for 48 layers twice in 56.7s

            RTN      AWQ     GPTQ   (FP32 = 56.51)
4-bit     66.15    64.11    60.21
3-bit    223.56   164.76    85.77


## 2.5 GGUF: CPU / Hybrid Quantization via llama.cpp

GGUF is a **file format** (not a quantization algorithm) designed for efficient storage and loading of quantized models, originally from the llama.cpp project.

### K-quants (Mixed Precision):
K-quants (introduced by "ikawrakow" in llama.cpp) use mixed precision: different layers get different bit-widths based on their importance. Examples include `Q4_K_M`, `Q5_K_M`, etc.

### Strengths:
- Runs on CPU (no GPU required!) — ideal for consumer hardware.
- Supports hybrid CPU+GPU inference (offload $N$ layers to GPU).
- Excellent ecosystem: llama.cpp, Ollama, LM Studio, GPT4All.

### Convert & quantize your own model to GGUF
(shell steps — they need a llama.cpp build, so they are not run here)
- 1. Clone llama.cpp:  git clone https://github.com/ggerganov/llama.cpp
- 2. Build:            cd llama.cpp && make
- 3. Convert:          python convert_hf_to_gguf.py --input /path/to/model --output model.gguf
- 4. Quantize:         ./llama-quantize model.gguf model-Q4_K_M.gguf Q4_K_M

Below we download a ready-made `Q4_K_M` GGUF of Qwen2.5-0.5B-Instruct and run it with `llama-cpp-python` (Metal on Apple Silicon, CUDA on NVIDIA, otherwise pure CPU).

In [14]:
import os
from llama_cpp import Llama

# Download a Q4_K_M GGUF straight from the Hub and load it
llm = Llama.from_pretrained(
    repo_id="Qwen/Qwen2.5-0.5B-Instruct-GGUF",
    filename="qwen2.5-0.5b-instruct-q4_k_m.gguf",
    n_ctx=2048,  # context window length
    n_gpu_layers=-1,  # offload every layer to the GPU if there is one (0 = pure CPU)
    verbose=False,
)

gguf_mb = os.path.getsize(llm.model_path) / 1e6
bf16_mb = 494_032_768 * 2 / 1e6  # Qwen2.5-0.5B parameter count × 2 bytes
print(f"GGUF Q4_K_M file : {gguf_mb:.0f} MB")
print(f"Same model, BF16 : {bf16_mb:.0f} MB  ({gguf_mb / bf16_mb:.0%} of the size)\n")

response = llm.create_chat_completion(
    messages=[{"role": "user", "content": "Explain the theory of relativity simply."}],
    max_tokens=120,
    temperature=0.0,
)
print(response["choices"][0]["message"]["content"])

del llm

GGUF Q4_K_M file : 491 MB
Same model, BF16 : 988 MB  (50% of the size)



The theory of relativity is a set of fundamental principles that describe the behavior of objects and systems in the universe, including the laws of motion, the laws of energy and mass, and the laws of gravity. These principles were developed by Albert Einstein in the early 20th century and have since become the foundation of modern physics.

The theory of relativity is based on the following key principles:

1. The laws of motion: The laws of motion describe the behavior of objects and systems in the universe, including the laws of motion, the laws of energy and mass, and the laws of gravity


## 2.6 BitsAndBytes: NF4 / INT8 Mixed-Precision

Library by Tim Dettmers (UW). Two key features:

### 8-bit (LLM.int8()):
- Decomposes matrix multiplication into two parts:
  - (a) Normal INT8 matmul for the majority of features
  - (b) FP16 matmul for "outlier" features (those with magnitude > 6.0)
- This handles the activation outlier problem that ruins naive INT8.

### 4-bit (QLoRA / NF4):
- **NF4** = "NormalFloat4" — a 4-bit data type whose quantization levels are optimally spaced for normally-distributed weights (information-theoretically optimal for Gaussian data).
- **Double quantization:** quantize the quantization constants themselves to save even more memory.
- Primarily designed for QLoRA (4-bit base model + LoRA adapters in FP16).

Recent `bitsandbytes` releases also run on CPU, so we can load Qwen2.5-0.5B-Instruct in 8-bit and NF4 and compare memory and output against BF16.

In [15]:
from transformers import BitsAndBytesConfig

bnb_model_name = "Qwen/Qwen2.5-0.5B-Instruct"
bnb_tokenizer = AutoTokenizer.from_pretrained(bnb_model_name)
bnb_device = "cuda" if torch.cuda.is_available() else "cpu"

configs = {
    "BF16": None,
    # 8-bit LLM.int8(): INT8 matmul + FP16 path for outlier features
    "INT8 (LLM.int8)": BitsAndBytesConfig(load_in_8bit=True),
    # 4-bit NF4 with double quantization, compute in BF16 (the QLoRA setup)
    "NF4": BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",  # NormalFloat4 (optimal for Gaussian weights)
        bnb_4bit_compute_dtype=torch.bfloat16,  # compute in BF16 during forward pass
        bnb_4bit_use_double_quant=True,  # quantize the quantization constants too
    ),
}

messages = [{"role": "user", "content": "What is the meaning of life? Answer in one sentence."}]
prompt = bnb_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

for label, config in configs.items():
    model_bnb = AutoModelForCausalLM.from_pretrained(
        bnb_model_name,
        # LLM.int8() runs its outlier path in FP16; the other configs use BF16
        dtype=torch.float16 if label.startswith("INT8") else torch.bfloat16,
        quantization_config=config,
        device_map=bnb_device,
    )
    inputs = bnb_tokenizer(prompt, return_tensors="pt").to(bnb_device)
    # (1, prompt_len) -> (1, prompt_len + max_new_tokens)
    outputs = model_bnb.generate(**inputs, max_new_tokens=40, do_sample=False)
    answer = bnb_tokenizer.decode(
        outputs[0, inputs["input_ids"].shape[1] :], skip_special_tokens=True
    )
    print(f"{label:16s} {model_bnb.get_memory_footprint() / 1e6:6.0f} MB → {answer}")
    del model_bnb

BF16                988 MB → The question "What is the meaning of life?" is deeply philosophical and subjective, with no single definitive answer that applies to all individuals or cultures. However, some people might believe that the purpose of existence


INT8 (LLM.int8)     630 MB → The question "What is the meaning of life?" is deeply philosophical and subjective. It's a fundamental inquiry that touches on various aspects of human existence, including existential questions about purpose, meaning, and the


NF4                 451 MB → The meaning of life is finding purpose and fulfillment through living one's values and making positive contributions to society.


## 2.7 SmoothQuant: W8A8 (Weight AND Activation Quantization)

SmoothQuant is typically applied through inference engines:
- NVIDIA TensorRT-LLM (native support)
- MIT-HAN-LAB/smoothquant (reference implementation)

### The Problem
Activation tensors have large **OUTLIERS** (some channels have values 10–100× larger than others). This makes direct INT8 quantization of activations very lossy. However, weights are relatively smooth and easy to quantize.

### Key Insight
"Migrate difficulty from activations to weights" by inserting a per-channel smoothing factor $s_j$ that divides activations and multiplies weights BEFORE quantization:

$$Y = (X \cdot \text{diag}(s)^{-1}) \cdot (\text{diag}(s) \cdot W)$$

Choose $s_j = \max(|X_j|)^\alpha / \max(|W_j|)^{1-\alpha}$, with $\alpha \in [0.5, 0.75]$.

### Result
Both weights AND activations can be quantized to INT8 (W8A8), enabling the use of efficient INT8 GEMM kernels on GPUs and achieving real hardware speedups.

Below we simulate W8A8 on GPT-2: weights per-output-channel INT8, activations **per-tensor** INT8 (one scale for the whole activation, which is what fast INT8 GEMM kernels want). We first look at the activation outliers, then compare perplexity with and without smoothing.

In [16]:
# How skewed are the activations? Per-channel max |X| going into block 6's QKV projection
# (num_tokens, in_dim) -> (in_dim,)
channel_max = layer_inputs["transformer.h.6.attn.c_attn"].abs().amax(dim=0)
print(f"Largest channel max : {channel_max.max():.2f}")
print(f"Median channel max  : {channel_max.median():.2f}")
print(f"Outlier channels    : {channel_max.topk(3).indices.tolist()}\n")


def fake_quant_int8(t, dim=None):
    """Symmetric INT8 quantize → dequantize. Per-tensor if dim is None, else per-slice along dim."""
    absmax = t.abs().max() if dim is None else t.abs().amax(dim=dim, keepdim=True)
    scale = absmax / 127
    return torch.clamp(torch.round(t / scale), -127, 127) * scale


def quantize_w8a8(model, smooth, alpha=0.5):
    model_q = deepcopy(model)
    for name in linear_names:
        layer = model_q.get_submodule(name)
        # (in_dim, out_dim)
        W = layer.weight.data
        # (in_dim,)
        s = torch.ones(W.shape[0])

        # SmoothQuant applies to layers right after a LayerNorm, where 1/s can be folded into the LayerNorm weights
        if smooth and name.endswith(("attn.c_attn", "mlp.c_fc")):
            # s_j = max|X_j|^α / max|W_j|^(1-α)
            act_scales = layer_inputs[name].abs().amax(dim=0).clamp(min=1e-5)
            weight_scales = W.abs().amax(dim=1).clamp(min=1e-5)
            s = act_scales.pow(alpha) / weight_scales.pow(1 - alpha)

        # Weights absorb the scale, then get per-output-channel INT8
        # (in_dim, out_dim)
        layer.weight.data = fake_quant_int8(W * s.unsqueeze(1), dim=0)

        # Activations are divided by s (outliers shrink), then get per-tensor INT8
        # (batch_num, seq_len, in_dim) -> (batch_num, seq_len, in_dim)
        layer.register_forward_pre_hook(lambda mod, inp, s=s: (fake_quant_int8(inp[0] / s),))
    return model_q


print(f"FP32                   perplexity: {ppl_results['FP32']:.2f}")
print(f"W8A8 naive             perplexity: {perplexity(quantize_w8a8(gpt2, smooth=False), eval_batch):.2f}")
for alpha in [0.5, 0.75]:
    ppl_smooth = perplexity(quantize_w8a8(gpt2, smooth=True, alpha=alpha), eval_batch)
    print(f"W8A8 SmoothQuant α={alpha} perplexity: {ppl_smooth:.2f}")

Largest channel max : 7.76
Median channel max  : 0.94
Outlier channels    : [64, 480, 266]

FP32                   perplexity: 56.51


W8A8 naive             perplexity: 68.77


W8A8 SmoothQuant α=0.5 perplexity: 64.52


W8A8 SmoothQuant α=0.75 perplexity: 63.75


# 3) Quantization-Aware Training (QAT)

All methods above are **Post-Training Quantization (PTQ)**: quantize after training is complete.

**QAT** integrates quantization INTO the training loop.

### How QAT works:
1. Insert "fake quantization" nodes after weights and/or activations.
2. During **FORWARD** pass: quantize → dequantize (simulates quantization noise).
3. During **BACKWARD** pass: use the "Straight-Through Estimator" (STE) to pass gradients through the non-differentiable rounding operation.
4. The model learns to be robust to quantization noise during training.

**Advantages:** Much better accuracy at very low bit-widths (2-bit, 1-bit).
**Disadvantages:** Requires full training (or at least fine-tuning) — expensive.

In [17]:
class FakeQuantize(torch.autograd.Function):
    """
    Fake quantization: quantize → dequantize in the forward pass,
    but pass gradients straight through in the backward pass (STE).

    This simulates quantization noise during training so the model
    learns weights that are robust to rounding.
    """

    @staticmethod
    def forward(ctx, x, num_bits=8):
        qmin, qmax = -(2 ** (num_bits - 1)), 2 ** (num_bits - 1) - 1
        scale = x.abs().max() / qmax if x.abs().max() > 0 else torch.tensor(1.0)

        # Quantize (round) then dequantize
        x_quant = torch.clamp(torch.round(x / scale), qmin, qmax)
        x_dequant = x_quant * scale
        return x_dequant

    @staticmethod
    def backward(ctx, grad_output):
        # Straight-Through Estimator: pass gradient unchanged
        return grad_output, None


fake_quantize = FakeQuantize.apply


class QATLinear(nn.Module):
    """
    A linear layer with fake quantization applied to weights during training.
    During the forward pass, weights are fake-quantized (quantize then immediately
    dequantize) so the model sees the quantization noise and adapts.

    At deployment time, you replace this with actual INT8 weights.
    """

    def __init__(self, in_features, out_features, num_bits=8):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)
        self.num_bits = num_bits

    def forward(self, x):
        # Fake-quantize the weights (simulate INT8 rounding noise)
        # (output_dim, input_dim) — same as self.linear.weight
        w_quant = fake_quantize(self.linear.weight, self.num_bits)

        # Standard linear operation with fake-quantized weights
        # (batch_num, *, in_features) → (batch_num, *, out_features)
        return nn.functional.linear(x, w_quant, self.linear.bias)


# Demo: compare outputs before/after fake quantization
print("\n--- QAT: Fake quantization demo ---")
torch.manual_seed(42)
layer_normal = nn.Linear(4, 4)
layer_qat = QATLinear(4, 4, num_bits=8)
# Copy same weights so we can compare
layer_qat.linear.weight = layer_normal.weight
layer_qat.linear.bias = layer_normal.bias

# (batch_num=2, in_features=4)
x = torch.randn(2, 4)

out_normal = layer_normal(x)
out_qat = layer_qat(x)

print(f"FP32 output:\n{out_normal}")
print(f"\nQAT output (fake-quantized weights):\n{out_qat}")
print(f"\nDifference: {(out_normal - out_qat).abs().max():.8f}")


--- QAT: Fake quantization demo ---
FP32 output:
tensor([[ 1.5908,  0.3082,  0.0429,  0.5384],
        [ 0.3457, -0.1485,  0.0749,  0.3781]], grad_fn=<AddmmBackward0>)

QAT output (fake-quantized weights):
tensor([[ 1.5925,  0.3090,  0.0452,  0.5366],
        [ 0.3468, -0.1474,  0.0747,  0.3791]], grad_fn=<AddmmBackward0>)

Difference: 0.00222528


## 3.1 Frontier: BitNet b1.58 — Ternary / 1-bit LLMs

**Papers:**
- Wang et al. (2023): "BitNet: Scaling 1-bit Transformers for LLMs"
- Ma et al. (2024): "BitNet b1.58: The Era of 1-bit LLMs"

The idea: instead of quantizing AFTER training, train the model FROM SCRATCH with extreme low-bit weights — specifically ternary weights {-1, 0, +1}.

### Why "1.58 bits"?
With 3 possible values, information content = $\log_2(3) \approx 1.58$ bits/weight.

### BitLinear Layer:
1. **Weight quantization**: $w_q = \text{RoundClip}(w / (|w|_{\text{mean}} + \epsilon), -1, 1)$
2. **Activation quantization**: $x_q = \text{Clip}(x \times Q_b / |x|_\infty, -Q_b, Q_b)$
3. **Matrix multiply**: uses only addition/subtraction (no FP multiply needed!)
4. **Output rescaling**: $y = y_q \times \beta \times (|x|_\infty / Q_b)$

This is a **FRONTIER** research direction — the model must be trained from scratch with `BitLinear` layers. You cannot simply convert an existing model.

In [18]:
class BitLinear(nn.Module):
    """
    Weights are constrained to ternary values {-1, 0, +1} during forward pass.
    Activations are quantized per-token to a fixed range.

    True BitNet trains from scratch with
    this layer replacing all nn.Linear in the transformer. Efficient inference
    requires custom low-bit kernels (e.g., bitnet.cpp).
    """

    def __init__(self, in_features, out_features, activation_bits=8):
        super().__init__()
        # Full-precision "latent" weights (updated by optimizer)
        self.weight = nn.Parameter(torch.randn(out_features, in_features) * 0.02)
        self.bias = None
        self.eps = 1e-5
        self.Qb = 2 ** (activation_bits - 1)  # e.g. 128 for 8-bit

    def ternary_quantize_weight(self, w):
        """
        Quantize weight to {-1, 0, +1} using absmean scaling.
        w_q = RoundClip(w / γ, -1, 1), where γ = mean(|w|)
        """
        gamma = w.abs().mean() + self.eps
        w_scaled = w / gamma
        # Round to nearest integer and clamp to {-1, 0, 1}
        # (output_dim, input_dim)
        w_q = torch.clamp(torch.round(w_scaled), -1, 1)
        return w_q, gamma

    def quantize_activation(self, x):
        """
        Per-token absmax activation quantization.
        x_q = Clip(x × Qb / |x|_∞, -Qb, Qb)
        """
        # Per-token absolute max
        # (batch_num, seq_len, 1)
        x_absmax = x.abs().max(dim=-1, keepdim=True).values + self.eps
        x_scale = self.Qb / x_absmax

        # Scale and clip
        # (batch_num, seq_len, model_dim)
        x_q = torch.clamp(x * x_scale, -self.Qb, self.Qb)
        return x_q, x_scale

    def forward(self, x):
        # Quantize weights to ternary {-1, 0, +1}
        w_q, gamma = self.ternary_quantize_weight(self.weight)

        # Quantize activations
        x_q, x_scale = self.quantize_activation(x)

        # Matrix multiply (with STE: pretend w_q gradients flow through)
        # (batch_num, seq_len, in_features) × (in_features, out_features) → (batch_num, seq_len, out_features)
        w_effective = w_q.detach() + self.weight - self.weight.detach()  # STE
        y = nn.functional.linear(x_q, w_effective)

        # Rescale output: undo activation scaling and apply weight scale
        # (batch_num, seq_len, out_features)
        y = y * gamma / x_scale
        return y


# Demo: BitLinear forward pass
print("--- BitLinear demo ---")
torch.manual_seed(0)
bit_layer = BitLinear(in_features=8, out_features=4)

# Inspect ternary weights
w_q, gamma = bit_layer.ternary_quantize_weight(bit_layer.weight)
print(f"Latent weight (FP32):\n{bit_layer.weight.data[:2, :4]}")
print(f"Ternary weight:\n{w_q[:2, :4]}")
print(f"Unique values: {w_q.unique().tolist()}")
print(f"Gamma (absmean scale): {gamma:.6f}")

# Forward pass
# (batch_num=2, seq_len=3, model_dim=8)
x = torch.randn(2, 3, 8)
y = bit_layer(x)
print(f"\nInput shape : {x.shape}")
print(f"Output shape: {y.shape}")
print(f"Output:\n{y[0]}")

--- BitLinear demo ---
Latent weight (FP32):
tensor([[-0.0225, -0.0230, -0.0050, -0.0087],
        [ 0.0064, -0.0253,  0.0070,  0.0062]])
Ternary weight:
tensor([[-1., -1., -0., -1.],
        [ 0., -1.,  0.,  0.]], grad_fn=<SliceBackward0>)
Unique values: [-1.0, -0.0, 1.0]
Gamma (absmean scale): 0.016957

Input shape : torch.Size([2, 3, 8])
Output shape: torch.Size([2, 3, 4])
Output:
tensor([[ 0.0075,  0.0122,  0.0188,  0.0018],
        [ 0.0622,  0.0007, -0.0728,  0.0347],
        [-0.0103, -0.0122,  0.0136, -0.0208]], grad_fn=<SelectBackward0>)


## Summary

| Method | Bits | Type | Target HW | Notes |
|---|---|---|---|---|
| Dtype casting | 16 | N/A | GPU | FP32 → BF16/FP16, no loss |
| Zero-Point | 8 | PTQ | Any | Simple, per-tensor scale |
| Absmax | 8 | PTQ | Any | Symmetric, no zero-point |
| GPTQ | 4 | PTQ | GPU | Hessian-guided, accurate |
| AWQ | 4 | PTQ | GPU | Activation-aware scaling |
| GGUF (K-quants) | 2–8 | PTQ | CPU+GPU | Mixed-precision, flexible |
| BitsAndBytes | 4/8 | PTQ | GPU | NF4 optimal for QLoRA |
| SmoothQuant | 8 | PTQ W8A8 | GPU | Quantizes activations too |
| QAT (general) | 4–8 | QAT | Any | Train-time, best quality |
| BitNet b1.58 | 1.58 | QAT/train | Custom | Ternary {-1, 0, +1} |

### Quick Decision Guide
- Need to run on CPU/laptop? → **GGUF (Q4_K_M)**
- GPU inference, max speed? → **AWQ + Marlin kernel**
- GPU inference, max quality? → **GPTQ** or **BitsAndBytes (NF4)**
- Fine-tuning on limited GPU? → **BitsAndBytes NF4 + QLoRA**
- Both weights AND activations? → **SmoothQuant (W8A8)**
- Research / next frontier? → **BitNet b1.58** (train from scratch)